# Pré-traitement des données

## Vue d'ensemble

Ce notebook transforme les données brutes de comptage vélo en un dataset enrichi prêt pour la modélisation.

**Données d'entrée :**
- `comptage-velo-donnees-compteurs.csv` — comptages horaires par compteur (Paris Open Data)
- `vacances-scolaires-2023-2026.csv` — calendrier des vacances scolaires
- `open-meteo-48.82N2.29E43m.csv` — données météo horaires (température, précipitations)

**Étapes du pipeline :**
1. Chargement et nettoyage des colonnes inutiles
2. Suppression des doublons et des valeurs aberrantes (> 1 500 passages/heure)
3. Enrichissement géographique (latitude, longitude, direction)
4. Fusion avec les données météo
5. Gestion des valeurs manquantes
6. Ajout de variables calendaires (heure, jour, mois, week-end, vacances scolaires)
7. Création de lags temporels (1 h, 24 h, 168 h) et moyenne glissante sur 3 h

**Sorties :**
- `data/processed/df_processed.csv` — dataset principal pour la modélisation
- `data/processed/sites_agg.parquet` — statistiques agrégées par site (Streamlit)
- `data/processed/counters.parquet` — liste des compteurs (Streamlit)
- `data/processed/typical_counter_weekday_hour.parquet` / `typical_counter_hour.parquet` — comptages typiques pour l'estimation des lags

In [1]:
import pandas as pd
from pathlib import Path

# Emplacement des fichiers sources
RAW_DATA_PATH = Path('../data/raw/comptage-velo-donnees-compteurs.csv')
VACANCES_PATH = Path('../data/raw/vacances-scolaires-2023-2026.csv')
WEATHER_PATH = Path('../data/raw/open-meteo-48.82N2.29E43m.csv')

OUTPUT_DIR = Path('../data/processed')
OUTPUT_CLEAN_DF = Path('../data/processed/df_processed.csv')


## Fonctions de prétraitement

In [2]:
def load_raw(velo_path=RAW_DATA_PATH, vacances_path=VACANCES_PATH, weather_path=WEATHER_PATH):
    """
    Cette fonction charge les fichiers sources et convertit les colonnes de dates en objets datetime.
    """
    # Importation du dataset principal
    df = pd.read_csv(velo_path, sep=';')

    # Suppression des colonnes inutiles
    col_a_supprimer = ['Identifiant du compteur', 'Identifiant du site de comptage',
                       "Date d'installation du site de comptage", 'Identifiant technique compteur', 'ID Photos',
                       'test_lien_vers_photos_du_site_de_comptage_', 'id_photo_1', 'url_sites', 'type_dimage', 'mois_annee_comptage']
    df.drop(
        col_a_supprimer,
        axis=1,
        inplace=True
    )

    # Conversion de 'Date et heure de comptage' en format datetime et heure locale
    df['Date et heure de comptage'] = (
        pd.to_datetime(df['Date et heure de comptage'], utc=True)
        .dt.tz_convert('Europe/Paris')
        .dt.tz_localize(None)
    )

    # Importation du dataset relatif aux vacances scolaires
    df_vacances_scolaires = pd.read_csv(
        vacances_path,
        sep=';',
        parse_dates=['Date'],
        dayfirst=True,
        encoding='latin-1'
    )

    # Importation du dataset avec des données météorologiques
    df_weather = pd.read_csv(weather_path, header=2)
    df_weather['Date et heure de comptage'] = (
        pd.to_datetime(df_weather['time'], utc=True)
        .dt.tz_convert('Europe/Paris')
        .dt.tz_localize(None)
    )
    df_weather.drop(
        'time',
        axis=1,
        inplace=True
    )
    df_weather = df_weather.rename(columns={
        'temperature_2m (°C)': 'Température (°C)',
        'precipitation (mm)': 'Précipitations (mm)'
    })

    return df, df_vacances_scolaires, df_weather

In [3]:
def remove_outliers(df):
    df = df[df['Comptage horaire'] < 1500]
    return df

In [4]:
def deduplicate(df):
    df = df.drop_duplicates(subset=['Nom du compteur', 'Date et heure de comptage'])
    return df

In [5]:
def add_direction(df):
    """
    Cette fonction crée une colonne avec le sens de la voie.
    """
    df['Direction'] = df['Nom du compteur'].str.extract(r'(Bike IN|Bike OUT|E-O|O-E|N-S|S-N|NE-SO|SO-NE|NO-SE|SE-NO)')
    return df

In [6]:
def coordinates(df):
    """
    Cette fonction extrait la latitude et la longitude de la colonne 'Coordonnées géographiques' et les convertit en valeurs numériques.
    """
    df[['Latitude', 'Longitude']] = df['Coordonnées géographiques'].str.split(',', expand=True)
    df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
    df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
    df.drop(
        'Coordonnées géographiques',
        axis=1,
        inplace=True
    )
    return df

In [7]:
def add_weather_features(df, df_weather):
    """
    Cette fonction enrichit le dataset principal avec deux nouvelles colonnes: Température (°C) et Précipitations (mm)
    """
    df = df.merge(
        df_weather,
        on='Date et heure de comptage',
        how='left'
    )
    return df

In [8]:
def fill_nan(df):
    """
    Cette fonction a pour objectif de gérer les valeurs manquantes, notamment celles identifiées lors de l'exploration des données.
    """
    # Compléter les lignes vides dans la colonne 'Nom du site de comptage'
    replace_nan = {
        "27 quai de la Tournelle": "27 quai de la Tournelle",
        "Grande Armée": "10 avenue de la Grande Armée",
        "Face au 48 quai de la marne": "Face au 48 quai de la marne",
        "Pont des Invalides": "Pont des Invalides",
        "Quai des Tuileries": "Quai des Tuileries",
        "Totem 64 Rue de Rivoli": "Totem 64 Rue de Rivoli"
    }
    for value, remplacement in replace_nan.items():
        mask = df['Nom du site de comptage'].isna() & df['Nom du compteur'].str.contains(value, na=False)
        df.loc[mask, 'Nom du site de comptage'] = remplacement

    # Compléter les lignes vides dans la colonne 'Liens vers photo du site de comptage'
    df['Lien vers photo du site de comptage'] = (
        df.groupby('Nom du site de comptage')['Lien vers photo du site de comptage']
        .transform(lambda row: row.ffill().bfill())
    )

    # Compléter les lignes potentiellement vides dans la colonne 'Direction'
    df['Direction'] = df['Direction'].fillna("N/A")

    # Compléter les lignes vides dans les colonnes 'Latitude' et 'Longitude'
    df['Latitude'] = df.groupby('Nom du site de comptage')['Latitude'].transform(lambda row: row.ffill().bfill())
    df['Longitude'] = df.groupby('Nom du site de comptage')['Longitude'].transform(lambda row: row.ffill().bfill())

    # Remplir les lignes vides dans les colonnes 'Température (°C)' et 'Précipitations (mm)' avec la moyenne du jour
    df['Date'] = df['Date et heure de comptage'].dt.date
    df_weather_means = df.groupby('Date')[['Température (°C)', 'Précipitations (mm)']].mean().rename(columns={
        'Température (°C)': 'Température_moy_jour',
        'Précipitations (mm)': 'Précipitations_moy_jour'
    })
    df = df.merge(
        df_weather_means,
        on='Date',
        how='left'
    )
    df['Température (°C)'] = df['Température (°C)'].fillna(df['Température_moy_jour'])
    df['Précipitations (mm)'] = df['Précipitations (mm)'].fillna(df['Précipitations_moy_jour'])
    df.drop(columns=['Date', 'Température_moy_jour', 'Précipitations_moy_jour'], inplace=True)

    df = df.dropna(subset=['Nom du compteur', 'Nom du site de comptage', 'Comptage horaire', 'Date et heure de comptage'])

    return df

In [9]:
def add_calendar_features(df, df_vacances_scolaires):
    """
    Cette fonction génère des features temporelles, ajoute un indicateur de vacances scolaires,
    puis crée des lags (1h, 24h, 168h) et une moyenne glissante sur 3 heures pour modéliser les effets temporels du comptage horaire.
    """
    # Création de variables calendaires à partir de la colonne 'Date et heure de comptage'
    df['Jour du mois'] = df['Date et heure de comptage'].dt.day
    df['Mois'] = df['Date et heure de comptage'].dt.month
    df['Année'] = df['Date et heure de comptage'].dt.year
    df['Heure'] = df['Date et heure de comptage'].dt.hour
    df['Jour de la semaine'] = df['Date et heure de comptage'].dt.dayofweek
    df['Week-end'] = (df['Date et heure de comptage'].dt.dayofweek >= 5).astype(int)

    # Création d'une variable 'Vacances'
    df['Date'] = pd.to_datetime(df['Date et heure de comptage']).dt.date
    df_vacances_scolaires['Date'] = pd.to_datetime(df_vacances_scolaires['Date'], dayfirst=True).dt.date
    vacances_set = set(df_vacances_scolaires['Date'])
    df['Vacances'] = (df['Date'].isin(vacances_set)).astype(int)
    df.drop(
        ['Date'],
        axis=1,
        inplace=True
    )

    # Création de variables de décalage (lags) et d'une moyenne glissante
    df = df.sort_values(['Nom du compteur', 'Date et heure de comptage']).copy()

    df['lag_1h'] = df.groupby('Nom du compteur')['Comptage horaire'].shift(1)
    df['lag_24h'] = df.groupby('Nom du compteur')['Comptage horaire'].shift(24)
    df['lag_168h'] = df.groupby('Nom du compteur')['Comptage horaire'].shift(168)

    df['roll_mean_3h'] = (
        df.groupby('Nom du compteur')['Comptage horaire']
        .shift(1)
        .rolling(window=3)
        .mean()
    )

    return df

In [10]:
def overwrite_data(df):
    """
    Cette fonction harmonise les informations de trois sites spécifiques en leur attribuant un seul jeu de coordonnées et une seule photo.
    """
    liens_ref = {
        'Pont National': 'https://filer.eco-counter-tools.com/file/2a/799b98880f593cad49159a30639b855596a7b8448fcb43d18a3eb2f84098772a/Y2H18086317_20200818152643.jpg',
        'Pont de la Concorde': 'https://filer.eco-counter-tools.com/file/3c/6241728bed2f3a14a9c830b39e0e3989fdd95ea2385d05d300fb671a2edbd73c/Y2H20083602_20211005103137.jpg',
        'Pont du Garigliano': 'https://filer.eco-counter-tools.com/file/45/80d66480bbd17c6f2408ee29594c5158d14b61e02adfd8612655ed223c798445/15977339543880.jpg'
    }
    for site, lien in liens_ref.items():
        df.loc[df['Nom du site de comptage'] == site, ['Lien vers photo du site de comptage']] = lien

    coordonnees_ref = {
        'Pont National': (48.82639, 2.38448),
        'Pont de la Concorde': (48.86373, 2.31973),
        'Pont du Garigliano': (48.83994, 2.26692)
    }
    for site, coordonnees in coordonnees_ref.items():
        df.loc[df['Nom du site de comptage'] == site, ['Latitude', 'Longitude']] = coordonnees

    return df

## Pipeline

In [11]:
df, df_vacances_scolaires, df_weather = load_raw()
df = remove_outliers(df)
df = deduplicate(df)
df = add_direction(df)
df = coordinates(df)
df = add_weather_features(df, df_weather)
df = fill_nan(df)
df = add_calendar_features(df, df_vacances_scolaires)
df = overwrite_data(df)

/var/folders/nb/4_hq95jn5mj5mf7p_fh_hl6h0000gn/T/ipykernel_83497/1491606291.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda row: row.ffill().bfill())


In [12]:
df.head()

,Nom du compteur,Nom du site de comptage,Comptage horaire,Date et heure de comptage,Lien vers photo du site de comptage,Direction,Latitude,Longitude,Température (°C),Précipitations (mm),...,Mois,Année,Heure,Jour de la semaine,Week-end,Vacances,lag_1h,lag_24h,lag_168h,roll_mean_3h
118195,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,0,2025-01-07 11:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,4.3,1.8,...,1,2025,11,1,0,0,NaN,NaN,NaN,NaN
118001,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,13,2025-01-07 12:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,5.4,0.0,...,1,2025,12,1,0,0,0.0,NaN,NaN,NaN
118731,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,50,2025-01-07 13:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.3,0.0,...,1,2025,13,1,0,0,13.0,NaN,NaN,NaN
118004,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,54,2025-01-07 14:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.8,0.5,...,1,2025,14,1,0,0,50.0,NaN,NaN,21.0
118198,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,33,2025-01-07 15:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.9,0.0,...,1,2025,15,1,0,0,54.0,NaN,NaN,39.0


In [13]:
# Vérification des valeurs manquantes après le preprocessing
df.isnull().sum()

Nom du compteur                            0
Nom du site de comptage                    0
Comptage horaire                           0
Date et heure de comptage                  0
Lien vers photo du site de comptage     9830
Direction                                  0
Latitude                                   0
Longitude                                  0
Température (°C)                           0
Précipitations (mm)                        0
Jour du mois                               0
Mois                                       0
Année                                      0
Heure                                      0
Jour de la semaine                         0
Week-end                                   0
Vacances                                   0
lag_1h                                   108
lag_24h                                 2569
lag_168h                               17977
roll_mean_3h                             322
dtype: int64

Rappel : Le site 35 boulevard de Ménilmontant n'a pas de photo.

In [14]:
for col in df.columns:
    print(col, ":", df[col].nunique())

Nom du compteur : 108
Nom du site de comptage : 66
Comptage horaire : 1274
Date et heure de comptage : 9858
Lien vers photo du site de comptage : 65
Direction : 10
Latitude : 65
Longitude : 66
Température (°C) : 363
Précipitations (mm) : 65
Jour du mois : 31
Mois : 12
Année : 2
Heure : 24
Jour de la semaine : 7
Week-end : 2
Vacances : 2
lag_1h : 1274
lag_24h : 1274
lag_168h : 1271
roll_mean_3h : 2961


## Export du dataset prétraité

Export de `df_processed.csv`, utilisé comme base pour les notebooks de modélisation.

In [15]:
# Export du dataframe final en format csv
df.to_csv(
    OUTPUT_CLEAN_DF,
    index=False,
    encoding='utf-8'
)

## Fichiers de support pour l'application Streamlit

Les cellules suivantes génèrent les fichiers parquet utilisés par l'application Streamlit :
- `sites_agg.parquet` : coordonnées et statistiques agrégées par site (carte d'accueil)
- `counters.parquet` : liste des compteurs disponibles (page Démo)
- `typical_counter_weekday_hour.parquet` / `typical_counter_hour.parquet` : comptages typiques pour l'estimation automatique des lags (page Démo)

In [16]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
df_sites_agg = (
    df.groupby('Nom du site de comptage', as_index=False)
    .agg(
        lat=('Latitude', 'mean'),
        lon=('Longitude', 'mean'),
        avg_hourly_count=('Comptage horaire', 'mean'),
        n_hours=('Comptage horaire', 'size'),
        image_url=('Lien vers photo du site de comptage', 'first')
    )
    .rename(columns={'Nom du site de comptage': 'site'})
)

df_sites_agg.to_parquet(OUTPUT_DIR / 'sites_agg.parquet', index=False)

In [18]:
df_counters = (
    pd.Series(df['Nom du compteur'].unique(), name='Nom du compteur')
    .sort_values()
    .reset_index(drop=True)
    .to_frame()
)

df_counters.to_parquet(OUTPUT_DIR / 'counters.parquet', index=False)

In [19]:
typical_counter_weekday_hour = (
    df.groupby(['Nom du compteur', 'Jour de la semaine', 'Heure'], as_index=False)
    .agg(typical_count=('Comptage horaire', 'mean'))
)

typical_counter_hour = (
    df.groupby(['Nom du compteur', 'Heure'], as_index=False)
    .agg(typical_count_hour=('Comptage horaire', 'mean'))
)

typical_counter_weekday_hour.to_parquet(OUTPUT_DIR / 'typical_counter_weekday_hour.parquet', index=False)
typical_counter_hour.to_parquet(OUTPUT_DIR / 'typical_counter_hour.parquet', index=False)